# HAMIC · TPU pretraining

Set `HAMIC_SOURCE`, then run the cells in order. Select a run configuration, press **Train**, and run the training cell. Pin the same source commit and run ID to resume on another session.

The local chips form one JAX data-parallel mesh. Each chip holds a copy of the model, and each packed batch is split by row. The XLA attention path is dense; `reference` is available for numerical checks.

In [ ]:
# Any Git ref works after "@": a branch (main), a tag or a commit SHA. Pin a SHA for
# a multi-session run so every session uses the same code (resume checks it).
HAMIC_SOURCE = "git+https://github.com/minhhhduc/HAMIC-LM@main"
HUB_REPO = "HaianCao/hamic-runs"  # private model repo: checkpoints under runs/<run_id>/
DATA_REPO = "HaianCao/hamic-data"  # private dataset repo: tokenized shards
XET_HIGH_PERFORMANCE = True  # read at first huggingface_hub import; restart to change

In [ ]:
import json
import os as _os
import shutil
import sys
import tarfile
import tempfile
import urllib.request
from pathlib import Path
from urllib.parse import quote

if "hamic_jax" in sys.modules:
    raise RuntimeError("hamic_jax is already imported. Restart the kernel before reinstalling.")
_os.environ["HF_XET_HIGH_PERFORMANCE"] = "1" if XET_HIGH_PERFORMANCE else "0"
requested_ref = HAMIC_SOURCE.rsplit("@", 1)[-1]
repo_url = HAMIC_SOURCE.removeprefix("git+").rsplit("@", 1)[0].removesuffix(".git")
owner_repo = repo_url.split("github.com/", 1)[-1]
api_url = f"https://api.github.com/repos/{owner_repo}"
work_root = Path(tempfile.gettempdir()) / "hamic-tpu"
with urllib.request.urlopen(
    f"{api_url}/commits/{quote(requested_ref, safe='')}", timeout=30
) as response:
    installed_commit = json.load(response)["sha"]
source_dir = work_root / "source" / installed_commit
if not (source_dir / "src" / "hamic_jax").is_dir():
    source_dir.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=source_dir.parent) as staging:
        archive = Path(staging) / "source.tar.gz"
        urllib.request.urlretrieve(f"{repo_url}/archive/{installed_commit}.tar.gz", archive)
        unpack = Path(staging) / "unpack"
        unpack.mkdir()
        with tarfile.open(archive) as tar:
            tar.extractall(unpack, filter="data")
        (top,) = unpack.iterdir()
        shutil.move(str(top), source_dir)
sys.path.insert(0, str(source_dir / "src"))
from hamic_jax.dependencies import ensure_dependencies

installed = ensure_dependencies()
if installed:
    print(f"Installed: {', '.join(installed)}")
import hamic_jax

if Path(hamic_jax.__file__).resolve().parent != (source_dir / "src" / "hamic_jax").resolve():
    raise RuntimeError("hamic_jax was imported from outside the pinned source tree")
hamic_ready = installed_commit
print(f"`hamic_jax` ready from `@{requested_ref}` at commit `{installed_commit[:12]}`")

In [ ]:
import os

# Run configurations reference presets by repository-relative paths, so work from the
# checkout of exactly the installed commit.
os.chdir(source_dir)
configs_dir = source_dir / "configs"
run_configs = sorted(
    p.relative_to(source_dir).as_posix()
    for p in (configs_dir / "train").rglob("*.yaml")
)
print(f"Working directory `{source_dir}` · {len(run_configs)} run configurations")

In [ ]:
from IPython.display import Markdown, display

try:
    import jax
except ImportError as exc:
    raise RuntimeError("Install jax[tpu] on the TPU host, then restart the kernel.") from exc

assert hamic_ready == installed_commit
if not hasattr(jax.lax, "pcast"):
    raise RuntimeError("Upgrade jax[tpu] to a release with jax.lax.pcast, then restart.")
backend = jax.default_backend()
if backend != "tpu":
    raise RuntimeError(f"Expected a TPU, found {backend!r}. Attach the TPU and rerun.")
cache_dir = work_root / "jax_cache"
cache_dir.mkdir(parents=True, exist_ok=True)
jax.config.update("jax_compilation_cache_dir", str(cache_dir))
jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
devices = jax.local_devices()
display(Markdown(
    f"TPU **{devices[0].device_kind}** · JAX `{jax.__version__}` · "
    f"{len(devices)} local device(s) · code `{installed_commit[:12]}`"
))

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    secret_label = "HF_KEY"
    secret_value = UserSecretsClient().get_secret(secret_label)
except ImportError:
    secret_value = os.environ.get("HF_KEY", "")

hub_token = (secret_value or "").strip()
if not hub_token:
    raise RuntimeError(
        "Add HF_KEY in Kaggle Secrets (Add-ons -> Secrets) or set HF_KEY in the environment, then rerun."
    )
os.environ["HF_KEY"] = hub_token
print("HF_KEY is configured.")

In [ ]:
from IPython.display import Markdown, display

from hamic_jax.checkpoint import HuggingFaceObjectStore

run_store = HuggingFaceObjectStore(HUB_REPO, private=True, token=hub_token)
data_store = HuggingFaceObjectStore(
    DATA_REPO, repo_type="dataset", create=False, token=hub_token
)
account = run_store.whoami()
display(Markdown(f"Hub account **{account}** · runs `{HUB_REPO}` · data `{DATA_REPO}`"))

In [ ]:
import ipywidgets as widgets
from IPython.display import display

from hamic_jax.presets import load_training_config

default = next((c for c in run_configs if "proxy" in c), run_configs[0])
config_choice = widgets.Dropdown(
    options=run_configs, value=default, description="Run configuration"
)
run_name = widgets.Text(placeholder="Default: config run ID + -tpu", description="TPU run ID")
mode = widgets.RadioButtons(
    options=[
        ("Test run (first steps only, nothing saved)", "bench"),
        ("Train / resume", "train"),
    ],
    value="bench",
    description="Mode",
)
# Test runs keep the full-length schedule and stop early.
test_steps = widgets.BoundedIntText(value=60, min=10, max=5000, step=10, description="Test steps")
test_lr = widgets.Text(value="", description="Test learning rate (empty: from config)")
test_warmup = widgets.Text(value="", description="Test warmup ratio (empty: from config)")
recompute_toggle = widgets.Checkbox(value=True, description="Recompute activations (16 GiB HBM)")
session_hours = widgets.BoundedFloatText(value=8.5, min=0.5, max=12.0, step=0.5, description="Session hours")
display(widgets.VBox([
    config_choice,
    run_name,
    mode,
    widgets.HBox([test_steps, test_lr, test_warmup]),
    widgets.HBox([recompute_toggle, session_hours]),
]))
print("Set the controls above, then run the remaining cells in order.")

In [ ]:
from dataclasses import replace

import pandas as pd
from IPython.display import display

from hamic_jax.runs import run_prefix
from hamic_jax.spec import read_yaml

config_path = source_dir / config_choice.value
config, preset = load_training_config(config_path)
run_id = run_name.value.strip() or f"{config.run_id}-tpu"
run_prefix(run_id)  # Validate before using this ID in local or Hub paths.
config = replace(config, run_id=run_id, recompute=recompute_toggle.value)
raw_config = {**read_yaml(config_path), "run_id": run_id}
summary = {
    "run_id": config.run_id,
    "model": f"{preset.name} ({preset.architecture})",
    "parameters": f"{preset.num_parameters / 1e6:.1f}M",
    "objective": config.objective
    + (f" (mask {config.mask_rate:.0%})" if config.objective == "mlm" else ""),
    "tokens": f"{config.total_tokens / 1e9:.3f}B",
    "steps": config.total_steps,
    "tokens_per_step": config.tokens_per_step,
    "tokenized": config.tokenized,
    "recompute": config.recompute,
    "sparse_prediction": config.sparse_prediction,
}
display(pd.DataFrame([summary]))

In [ ]:
from IPython.display import Markdown, display

from hamic_jax.data import fetch_tokenized, tokenized_prefix

prefix = tokenized_prefix(config.tokenized)
local_tokens = work_root / "corpora" / prefix
corpus = fetch_tokenized(data_store, prefix, local_tokens)
config = replace(config, tokenized=str(local_tokens))
split_tokens = {name: value["tokens"] for name, value in corpus.manifest["splits"].items()}
epochs = config.total_tokens / split_tokens["train"]
display(Markdown(
    f"Shards ready: `{split_tokens}` · the budget is **{epochs:.2f} epochs** of train"
))

In [ ]:
import ipywidgets as widgets
from IPython.display import display

train_button = widgets.Button(description="Train", button_style="success")
train_requested = {"value": False}


def _request_train(_):
    train_requested["value"] = True


train_button.on_click(_request_train)
display(train_button)

In [ ]:
from IPython.display import Markdown, display

from hamic_jax.lab import run_training_session
from hamic_jax.session import SessionLimits, run_session

if not train_requested["value"]:
    raise RuntimeError("Press Train to start.")
train_requested["value"] = False

if mode.value == "bench":
    overrides = {}
    if test_lr.value.strip():
        overrides["learning_rate"] = float(test_lr.value)
    if test_warmup.value.strip():
        overrides["warmup_ratio"] = float(test_warmup.value)
    run_config = replace(config, **overrides)
    print(
        f"Test run: lr={run_config.learning_rate:g}, "
        f"warmup={run_config.warmup_steps} steps, stop after {test_steps.value}",
        flush=True,
    )
    result = run_session(
        run_config,
        preset,
        corpus,
        None,
        work_root / "bench",
        SessionLimits(
            steps=test_steps.value,
            checkpoint_every_steps=0,
            checkpoint_every_seconds=0.0,
            log_every_steps=max(1, test_steps.value // 5),
            eval_every_steps=0,
            checkpoint_at_end=False,
        ),
        source_commit=installed_commit,
        publish=False,
        log=print,
    )
else:
    result = run_training_session(
        config,
        preset,
        corpus,
        run_store,
        work_root / "runs" / config.run_id,
        seconds=session_hours.value * 3600.0,
        source_commit=installed_commit,
        handle_preemption=True,
        raw_config=raw_config,
        log=print,
    )

# Drop early log windows affected by compilation and warmup.
start = result.resumed_from_step or 0
rates = [
    record["tokens_per_second"] for record in result.metrics
    if record["step"] > start and "tokens_per_second" in record
]
report = (
    f"Status **{result.status}** · step `{result.final_step}/{config.total_steps}` · "
    f"resumed from `{result.resumed_from_step}` · published `{result.published_steps}`"
)
if rates:
    steady = rates[len(rates) // 2:]
    rate = sum(steady) / len(steady)
    hours_left = (config.total_steps - result.final_step) * config.tokens_per_step / rate / 3600
    report += f"\n\nThroughput **{rate:,.0f} tokens/s** · about **{hours_left:.1f} h** left"
report += "".join(f"\n\n> Warning: {warning}" for warning in result.warnings)
display(Markdown(report))

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

from hamic_jax.plots import loss_curve

if result.losses:
    figure = loss_curve(result)
    display(figure)
    plt.close(figure)

rows = [
    {
        "step": step,
        "eval_loss": loss,
        "eval_bits_per_byte": result.eval_bits_per_byte.get(step),
    }
    for step, loss in sorted(result.evaluations.items())
]
if rows:
    display(pd.DataFrame(rows))
else:
    print("No evaluations in this session.")